# Raw Ingestion
## CSV → Delta Lake Tables

**Objectif :** Charger les fichiers CSV sources dans des tables Delta brutes.

**Principe d'immuabilité :** Les tables raw ne sont jamais modifiées après ingestion.
Toute transformation se fait en staging.

**Source :** `/Volumes/sport_metrics/raw/files/`

| Fichier CSV | Table Delta | Lignes |
|---|---|---|
| team_players_stats.csv | raw.team_players_stats | 5004 |
| team_training_sessions.csv | raw.team_training_sessions | 9464 |
| team_players_personal_info.csv | raw.team_players_personal_info | 60 |
| team_games_dataset.csv | raw.team_games_dataset | 389 |
| team_boxscores.csv | raw.team_boxscores | 389 |

**Choix technique :** `read_files()` avec `inferSchema = true`
→ Détection automatique des types · Pas de DDL manuel · Reproductible

In [0]:
-- ============================================
-- Ingestion : team_players_stats
-- Clé : PLAYER_ID + GAME_ID
-- Usage : staging → mart_player · mart_physical_condition
-- ============================================
CREATE TABLE IF NOT EXISTS sport_metrics.raw.team_players_stats
USING DELTA
AS SELECT * FROM read_files(
    '/Volumes/sport_metrics/raw/files/team_players_stats.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
-- ============================================
-- Ingestion : team_training_sessions
-- Clé : SESSION_ID
-- Usage : staging → int_fatigue_index → mart_physical_condition
-- Table la plus volumineuse
-- ============================================
CREATE TABLE IF NOT EXISTS sport_metrics.raw.team_training_sessions
USING DELTA
AS SELECT * FROM read_files(
    '/Volumes/sport_metrics/raw/files/team_training_sessions.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
-- ============================================
-- Ingestion : team_players_personal_info
-- Clé : PLAYER_ID
-- Usage : staging → dim_players_info
-- Table de référence : 60 joueurs
-- ============================================
CREATE TABLE IF NOT EXISTS sport_metrics.raw.team_players_personal_info
USING DELTA
AS SELECT * FROM read_files(
    '/Volumes/sport_metrics/raw/files/team_players_personal_info.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
-- ============================================
-- Ingestion : team_games_dataset
-- Clé : GAME_ID
-- Usage : staging → mart_game · dim_calendrier_matchs
-- Particularité : dates en français → parse_date_fr en staging
-- ============================================
CREATE TABLE IF NOT EXISTS sport_metrics.raw.team_games_dataset
USING DELTA
AS SELECT * FROM read_files(
    '/Volumes/sport_metrics/raw/files/team_games_dataset.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
-- ============================================
-- Ingestion : team_boxscores
-- Clé : GAME_ID
-- Usage : raw uniquement → jointure directe dans staging.team_games_dataset
-- Pas de staging dédié consommé directement en raw
-- ============================================
CREATE TABLE IF NOT EXISTS sport_metrics.raw.team_boxscores
USING DELTA
AS SELECT * FROM read_files(
    '/Volumes/sport_metrics/raw/files/team_boxscores.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
-- ============================================
-- Vérification : nombre de lignes par table
-- ============================================
SELECT 'team_players_stats'        AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.raw.team_players_stats
UNION ALL
SELECT 'team_training_sessions'    AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.raw.team_training_sessions
UNION ALL
SELECT 'team_players_personal_info'AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.raw.team_players_personal_info
UNION ALL
SELECT 'team_games_dataset'        AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.raw.team_games_dataset
UNION ALL
SELECT 'team_boxscores'            AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.raw.team_boxscores
ORDER BY nb_lignes DESC;